## HDDM analysis notebook for ` judgment type`

## Core Research Question
This analysis investigates the computational mechanism of judgment by determining the latent locus of the `judgment_condition × pair_type` interaction:

* **$v$ (Drift Rate):** The judgment effect is driven by changes in the efficiency of evidence accumulation (perceptual representation).
* **$a$ (Boundary Separation):** The judgment effect is driven by changes in decision caution (strategic adjustment).
* **Both:** Judgment exerts a joint influence on both representation and strategy.

## Design assumptions
- Original task is 2-choice, but HDDM here is fit on **correct (`1`) vs incorrect (`0`)**
- `pair_type` blank / missing cells are treated as **`None`**
- This notebook focuses on **`a` and `v`**, not `z`

## Recommended running order
1. Run the imports and settings cells
2. Run the helper-functions cell once
3. Run the data-loading / cleaning section
4. Inspect descriptives and plots
5. Define and inspect the model space
6. Fit **one test model**
7. Fit a **small subset**
8. Fit the **full model space**
9. Compare DIC and inspect the winning model


## Before you start

Please edit these items in **User settings**:

- `INPUT_PATH`
- `OUTPUT_DIR`
- sampling parameters for test vs formal runs

For a first smoke test, keep the test sampling settings small.


## 1. Imports


In [ ]:

from pathlib import Path
import json
import re
from typing import Dict, List, Tuple, Optional
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.environ['PYTHONHASHSEED'] = "123"
random.seed(123)
np.random.seed(123)
try:
    import hddm
except ImportError as exc:
    raise ImportError(
        "Could not import hddm. Please install HDDM in the current environment first."
    ) from exc


## 2. User settings


In [ ]:

# =========================
# User settings: edit here
# =========================
INPUT_PATH = Path("../../data/hddm/xab_main_trial_level_combined.csv")
OUTPUT_DIR = Path("../../results/hddm")

# RT trimming for HDDM input
MIN_RT = 0.20
MAX_RT = 3

# Quick test settings
SAMPLES_TEST = 100
BURN_TEST = 10
THIN_TEST = 1

# More formal settings
SAMPLES_FULL = 15000
BURN_FULL = 5000
THIN_FULL = 2

# HDDM outlier parameter
P_OUTLIER = 0.05

# Optional: manually set references if you want fixed baselines
MANUAL_JUDGMENT_REFERENCE = "categorical"   # e.g., "categoricall"
MANUAL_PAIR_REFERENCE = "between"    # e.g., "betwee"


## 3. Helper functions and constants

Run this cell **once**. It defines preprocessing, plotting, model-space construction, and model-fitting functions.


In [ ]:

# ==========================================
# Experimental Design Configuration
# ==========================================
# Constants reflecting the required structure of the combined dataset.
REQUIRED_COLUMNS = {"judgment_condition", "pair_type", "xab_resp", "xab_rt", "subj_idx"}
PREFERRED_PAIR_ORDER = ["within", "between"]
PREFERRED_JUDGMENT_ORDER = ["categorical", "continuous"]


# ==========================================
# Helper Functions
# ==========================================

def ensure_output_dir(path: Path) -> Path:
    """
    Ensures the existence of the specified output directory, creating it if necessary.

    Args:
        path (Path): The target directory path.

    Returns:
        Path: The validated and existing directory path.
    """
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def read_data(path: Path) -> pd.DataFrame:
    """
    Reads behavioral data from a given file path based on its extension.

    Args:
        path (Path): The path to the data file.

    Returns:
        pd.DataFrame: The loaded dataset.

    Raises:
        ValueError: If the file extension is not supported (.xlsx, .xls, .csv).
    """
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported input file type: {suffix}")
    

def pick_subject_column(df: pd.DataFrame) -> str:
    """
    Identifies the subject identifier column from the dataframe.

    Args:
        df (pd.DataFrame): The dataset to inspect.

    Returns:
        str: The name of the subject column.

    Raises:
        ValueError: If no standard subject identifier column is found.
    """
    for col in ["subj_idx"]:
        if col in df.columns:
            return col
    raise ValueError(
        "Could not find a subject identifier column. Expected one of: "
        "subjID, subjid, participant, subject, Subject."
    )


def sanitize_label_for_filename(text: str) -> str:
    """
    Sanitizes a string to ensure it is safe to use as a filename.

    Args:
        text (str): The raw label/string.

    Returns:
        str: A sanitized string containing only alphanumeric characters, dots, 
             underscores, or hyphens.
    """
    text = str(text)
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", text)
    return text.strip("_")


def save_current_fig(output_dir: Path, filename: str, dpi: int = 300) -> None:
    """
    Saves the currently active matplotlib figure to the specified directory.

    Args:
        output_dir (Path): The target directory for saving the figure.
        filename (str): The name of the output image file.
        dpi (int, optional): The resolution in dots per inch. Defaults to 300.
    """
    output_dir = ensure_output_dir(output_dir)
    plt.gcf().tight_layout()
    plt.gcf().savefig(output_dir / filename, dpi=dpi)


def ordered_present_levels(values: pd.Series, preferred_order: List[str]) -> List[str]:
    """
    Extracts and orders categorical levels present in the data based on a predefined preference.

    Args:
        values (pd.Series): The column containing categorical data.
        preferred_order (List[str]): The desired order of categories.

    Returns:
        List[str]: The ordered list of unique levels present in the series.
    """
    present = [str(v) for v in values.dropna().unique().tolist()]
    ordered = [lvl for lvl in preferred_order if lvl in present]
    leftovers = sorted([lvl for lvl in present if lvl not in ordered])
    return ordered + leftovers


def choose_pair_reference(levels: List[str]) -> str:
    """
    Determines the reference level for pair types based on experimental preference.

    Args:
        levels (List[str]): Available pair type levels.

    Returns:
        str: The selected reference level.
    """
    for lvl in PREFERRED_PAIR_ORDER:
        if lvl in levels:
            return lvl
    return levels[0]


def stderr(x: pd.Series) -> float:
    """
    Calculates the standard error of the mean (SEM) for a given sequence.

    Args:
        x (pd.Series): A sequence of numeric values.

    Returns:
        float: The standard error, or np.nan if the length is insufficient.
    """
    x = pd.Series(x).dropna()
    if len(x) <= 1:
        return np.nan
    return x.std(ddof=1) / np.sqrt(len(x))


# ==========================================
# Core Processing & Analysis Functions
# ==========================================

def prepare_hddm_data(
    df: pd.DataFrame,
    min_rt: float,
    max_rt: float,
    output_dir: Path,
) -> Tuple[pd.DataFrame, Dict[str, object]]:
    """
    Preprocesses and standardizes raw behavioral data for Hierarchical Drift 
    Diffusion Model (HDDM) analysis.

    Args:
        df (pd.DataFrame): The raw combined dataset.
        min_rt (float): The lower bound threshold for reaction times (in seconds).
        max_rt (float): The upper bound threshold for reaction times (in seconds).
        output_dir (Path): The directory to save the processed data and summary logs.

    Returns:
        Tuple[pd.DataFrame, Dict[str, object]]: A tuple containing the HDDM-formatted 
        dataframe and a dictionary summarizing data loss and preprocessing metrics.
        
    Raises:
        ValueError: If mandatory columns are missing from the dataframe.
    """
    output_dir = ensure_output_dir(output_dir)

    missing = REQUIRED_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    work = df.copy()

    # Exclude trials with missing critical experimental information
    work = work.dropna(subset=["subj_idx", "judgment_condition", "xab_resp", "xab_rt"]).copy()

    # Standardize 'judgment_condition' nomenclature
    work["judgment_condition"] = work["judgment_condition"].astype(str).str.strip().str.lower()

    # Standardize 'pair_type' and recode missing or empty values to "unknown"
    raw_pair = work["pair_type"].copy()
    pair_missing_mask = raw_pair.isna()
    raw_pair_str = raw_pair.astype(str).str.strip().str.lower()
    pair_blank_string_mask = raw_pair_str.isin(["", "nan", "none"])
    
    work["pair_type"] = raw_pair_str
    work.loc[pair_missing_mask | pair_blank_string_mask, "pair_type"] = "unknown"

    # Enforce numeric typing for dependent variables (response and RT)
    work["xab_resp"] = pd.to_numeric(work["xab_resp"], errors="coerce")
    work["xab_rt"] = pd.to_numeric(work["xab_rt"], errors="coerce")
    work = work.dropna(subset=["xab_resp", "xab_rt"]).copy()
    work = work[work["xab_resp"].isin([0, 1])].copy()

    # Apply Reaction Time (RT) truncation bounds to eliminate extreme outliers
    n_before_rt = len(work)
    work = work[(work["xab_rt"] >= min_rt) & (work["xab_rt"] <= max_rt)].copy()
    n_after_rt = len(work)

    # Establish categorical levels and reference points
    judgment_levels = ordered_present_levels(work["judgment_condition"], PREFERRED_JUDGMENT_ORDER)
    pair_levels = ordered_present_levels(work["pair_type"], PREFERRED_PAIR_ORDER)
    
    judgment_reference = (PREFERRED_JUDGMENT_ORDER[0] 
                          if PREFERRED_JUDGMENT_ORDER[0] in judgment_levels 
                          else judgment_levels[0])
    pair_reference = choose_pair_reference(pair_levels)

    work["judgment_condition"] = pd.Categorical(
        work["judgment_condition"], categories=judgment_levels, ordered=True
    )
    work["pair_type"] = pd.Categorical(
        work["pair_type"], categories=pair_levels, ordered=True
    )

    # Rename variables to comply with HDDM mandatory naming conventions
    hddm_df = work.rename(
        columns={"xab_rt": "rt", "xab_resp": "response"}
    ).copy()
    
    # Cast indexing and response columns to integers for modeling stability
    hddm_df["subj_idx"] = hddm_df["subj_idx"].astype(int)
    hddm_df["response"] = hddm_df["response"].astype(int)
    
    # Create an aggregate condition column for explicit interaction handling
    hddm_df["condition"] = (
        hddm_df["judgment_condition"].astype(str) + "_" + hddm_df["pair_type"].astype(str)
    )

    # Compile dataset summary for methodological reporting
    summary = {
        "n_trials_before_rt_filter": int(n_before_rt),
        "n_trials_after_rt_filter": int(n_after_rt),
        "n_trials_removed_by_rt_filter": int(n_before_rt - n_after_rt),
        "n_subjects": int(hddm_df["subj_idx"].nunique()),
        "judgment_levels": judgment_levels,
        "pair_levels": pair_levels,
        "judgment_reference": judgment_reference,
        "pair_reference": pair_reference,
        "n_pair_type_missing_recoded_to_unknown": int((pair_missing_mask | pair_blank_string_mask).sum()),
        "min_rt": float(min_rt),
        "max_rt": float(max_rt),
    }

    # Export preprocessing logs and formatted data
    with open(output_dir / "data_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    hddm_df.to_csv(output_dir / "hddm_ready_data.csv", index=False)
    return hddm_df, summary


def compute_descriptives(df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    """
    Computes summary descriptive statistics (trial counts, accuracy, and correct RTs) 
    aggregated at the subject and condition levels.

    Args:
        df (pd.DataFrame): The preprocessed dataset.

    Returns:
        Dict[str, pd.DataFrame]: A dictionary containing multiple dataframes summarizing 
        various descriptive metrics.
    """
    counts = (
        df.groupby(["judgment_condition", "pair_type"], observed=True)
        .size()
        .rename("n_trials")
        .reset_index()
    )

    subj_acc = (
        df.groupby(["subj_idx", "judgment_condition", "pair_type"], observed=True)["response"]
        .mean()
        .rename("accuracy")
        .reset_index()
    )

    acc_summary = (
        subj_acc.groupby(["judgment_condition", "pair_type"], observed=True)["accuracy"]
        .agg(mean="mean", se=stderr, n="count")
        .reset_index()
    )

    subj_rt_correct = (
        df[df["response"] == 1]
        .groupby(["subj_idx", "judgment_condition", "pair_type"], observed=True)["rt"]
        .mean()
        .rename("mean_rt_correct")
        .reset_index()
    )

    rt_summary = (
        subj_rt_correct.groupby(["judgment_condition", "pair_type"], observed=True)["mean_rt_correct"]
        .agg(mean="mean", se=stderr, n="count")
        .reset_index()
    )

    return {
        "counts": counts,
        "subj_acc": subj_acc,
        "acc_summary": acc_summary,
        "subj_rt_correct": subj_rt_correct,
        "rt_summary": rt_summary,
    }


def save_descriptives(desc: Dict[str, pd.DataFrame], output_dir: Path) -> None:
    """
    Exports the computed descriptive statistics dataframes to CSV files.

    Args:
        desc (Dict[str, pd.DataFrame]): Dictionary mapping descriptive names to their dataframes.
        output_dir (Path): The target directory for the exports.
    """
    output_dir = ensure_output_dir(output_dir)
    for name, table in desc.items():
        table.to_csv(output_dir / f"descriptives_{name}.csv", index=False)


def plot_trial_counts(counts: pd.DataFrame, output_dir: Optional[Path] = None):
    """
    Generates a bar plot visualizing trial distributions across conditions.

    Args:
        counts (pd.DataFrame): DataFrame containing aggregated trial counts.
        output_dir (Optional[Path]): Directory to save the figure, if specified.

    Returns:
        Tuple[plt.Figure, plt.Axes]: The matplotlib figure and axes objects.
    """
    fig, ax = plt.subplots()
    pivot_counts = counts.pivot(index="pair_type", columns="judgment_condition", values="xab_trial")
    pivot_counts.plot(kind="bar", ax=ax)
    
    ax.set_title("Trial Counts by Condition")
    ax.set_xlabel("Pair Type")
    ax.set_ylabel("Number of Trials")
    ax.legend(title="Judgment Condition")
    
    fig.tight_layout()
    if output_dir is not None:
        fig.savefig(Path(output_dir) / "plot_trial_counts.png", dpi=300)
    return fig, ax


def plot_accuracy(
    subj_acc: pd.DataFrame, 
    acc_summary: pd.DataFrame, 
    df: pd.DataFrame, 
    output_dir: Optional[Path] = None
):
    """
    Visualizes behavioral accuracy across conditions, displaying both individual 
    subject trajectories and group-level means with standard errors.

    Args:
        subj_acc (pd.DataFrame): Accuracy averaged per subject and condition.
        acc_summary (pd.DataFrame): Group-level accuracy summaries.
        df (pd.DataFrame): The core dataset for categorical level extraction.
        output_dir (Optional[Path]): Directory to save the figure, if specified.

    Returns:
        Tuple[plt.Figure, plt.Axes]: The matplotlib figure and axes objects.
    """
    fig, ax = plt.subplots()
    pair_order = [lvl for lvl in df["pair_type"].cat.categories if lvl in subj_acc["pair_type"].unique()]
    judgment_order = [lvl for lvl in df["judgment_condition"].cat.categories if lvl in subj_acc["judgment_condition"].unique()]
    
    x_positions = np.arange(len(pair_order))
    offsets = np.linspace(-0.08, 0.08, len(judgment_order))

    for r_idx, judgment_level in enumerate(judgment_order):
        subset = subj_acc[subj_acc["judgment_condition"] == judgment_level]
        wide = subset.pivot(index="subj_idx", columns="pair_type", values="accuracy")
        
        # Plot individual subject lines (faint)
        for _, row in wide.iterrows():
            ys = [row.get(n, np.nan) for n in pair_order]
            ax.plot(x_positions + offsets[r_idx], ys, alpha=0.15, linewidth=0.8)

        # Plot group-level means and standard errors (bold)
        means = acc_summary[acc_summary["judgment_condition"] == judgment_level].set_index("pair_type")
        ys = [means.loc[n, "mean"] if n in means.index else np.nan for n in pair_order]
        ses = [means.loc[n, "se"] if n in means.index else np.nan for n in pair_order]
        
        ax.errorbar(
            x_positions + offsets[r_idx],
            ys,
            yerr=ses,
            marker="o",
            linewidth=2,
            capsize=4,
            label=judgment_level,
        )

    ax.set_xticks(x_positions)
    ax.set_xticklabels(pair_order)
    ax.set_ylim(0, 1)
    ax.set_title("Accuracy by Condition")
    ax.set_xlabel("Pair Type")
    ax.set_ylabel("Proportion Correct")
    ax.legend(title="Judgment Condition")
    
    fig.tight_layout()
    if output_dir is not None:
        fig.savefig(Path(output_dir) / "plot_accuracy_by_condition.png", dpi=300)
    return fig, ax


def plot_correct_rt(
    subj_rt_correct: pd.DataFrame, 
    rt_summary: pd.DataFrame, 
    df: pd.DataFrame, 
    output_dir: Optional[Path] = None
):
    """
    Visualizes correct-trial reaction times across conditions, displaying both 
    individual subject trajectories and group-level means with standard errors.

    Args:
        subj_rt_correct (pd.DataFrame): Mean RT for correct trials per subject.
        rt_summary (pd.DataFrame): Group-level RT summaries.
        df (pd.DataFrame): The core dataset for categorical level extraction.
        output_dir (Optional[Path]): Directory to save the figure, if specified.

    Returns:
        Tuple[plt.Figure, plt.Axes]: The matplotlib figure and axes objects.
    """
    fig, ax = plt.subplots()
    pair_order = [lvl for lvl in df["pair_type"].cat.categories if lvl in subj_rt_correct["pair_type"].unique()]
    judgment_order = [lvl for lvl in df["judgment_condition"].cat.categories if lvl in subj_rt_correct["judgment_condition"].unique()]
    
    x_positions = np.arange(len(pair_order))
    offsets = np.linspace(-0.08, 0.08, len(judgment_order))

    for r_idx, judgment_level in enumerate(judgment_order):
        subset = subj_rt_correct[subj_rt_correct["judgment_condition"] == judgment_level]
        wide = subset.pivot(index="subj_idx", columns="pair_type", values="mean_rt_correct")
        
        # Plot individual subject lines (faint)
        for _, row in wide.iterrows():
            ys = [row.get(n, np.nan) for n in pair_order]
            ax.plot(x_positions + offsets[r_idx], ys, alpha=0.15, linewidth=0.8)

        # Plot group-level means and standard errors (bold)
        means = rt_summary[rt_summary["judgment_condition"] == judgment_level].set_index("pair_type")
        ys = [means.loc[n, "mean"] if n in means.index else np.nan for n in pair_order]
        ses = [means.loc[n, "se"] if n in means.index else np.nan for n in pair_order]
        
        ax.errorbar(
            x_positions + offsets[r_idx],
            ys,
            yerr=ses,
            marker="o",
            linewidth=2,
            capsize=4,
            label=judgment_level,
        )

    ax.set_xticks(x_positions)
    ax.set_xticklabels(pair_order)
    ax.set_title("Correct-Trial RT by Condition")
    ax.set_xlabel("Pair Type")
    ax.set_ylabel("Mean RT (s)")
    ax.legend(title="Judgment Condition")
    
    fig.tight_layout()
    if output_dir is not None:
        fig.savefig(Path(output_dir) / "plot_rt_correct_by_condition.png", dpi=300)
    return fig, ax


def build_model_space(judgment_ref: str, pair_ref: str) -> Dict[str, Dict[str, str]]:
    """
    Constructs a dictionary specifying a suite of plausible HDDM regression models, 
    parameterizing drift rate (v) and decision bound (a) through patsy-style formulas.

    Args:
        judgment_ref (str): The reference level for the judgment_condition factor.
        pair_ref (str): The reference level for the pair_type factor.

    Returns:
        Dict[str, Dict[str, str]]: Dictionary containing formula mappings and hypotheses 
        for each model in the model space.
    """
    train = f"C(judgment_condition, Treatment('{judgment_ref}'))"
    pair = f"C(pair_type, Treatment('{pair_ref}'))"

    return {
        "M0_baseline": {
            "a": "a ~ 1",
            "v": "v ~ 1",
            "hypothesis": "Null model: No condition effects on boundary (a) or drift rate (v).",
        },
        "M1_v_train_main": {
            "a": "a ~ 1",
            "v": f"v ~ 1 + {train}",
            "hypothesis": "Judgment type modulates drift rate exclusively.",
        },
        "M2_a_train_main": {
            "a": f"a ~ 1 + {train}",
            "v": "v ~ 1",
            "hypothesis": "Judgment type modulates decision boundary exclusively.",
        },
        "M3_av_train_main": {
            "a": f"a ~ 1 + {train}",
            "v": f"v ~ 1 + {train}",
            "hypothesis": "Judgment type modulates both boundary and drift rate independently.",
        },
        "M4_v_judgment_x_pair": {
            "a": "a ~ 1",
            "v": f"v ~ 1 + {train} * {pair}",
            "hypothesis": "Interaction between judgment and pair types manifests in drift rate.",
        },
        "M5_a_judgment_x_pair": {
            "a": f"a ~ 1 + {train} * {pair}",
            "v": "v ~ 1",
            "hypothesis": "Interaction between judgment and pair types manifests in decision boundary.",
        },
        "M6_v_interaction_a_train": {
            "a": f"a ~ 1 + {train}",
            "v": f"v ~ 1 + {train} * {pair}",
            "hypothesis": "Interaction is isolated to drift rate; boundary only carries generic judgment effect.",
        },
        "M7_a_interaction_v_train": {
            "a": f"a ~ 1 + {train} * {pair}",
            "v": f"v ~ 1 + {train}",
            "hypothesis": "Interaction is isolated to boundary; drift rate only carries generic judgment effect.",
        },
        "M8_av_judgment_x_pair": {
            "a": f"a ~ 1 + {train} * {pair}",
            "v": f"v ~ 1 + {train} * {pair}",
            "hypothesis": "Full interaction model: Both drift rate and boundary reflect interaction effects.",
        },
    }


def fit_single_model(
    data: pd.DataFrame,
    model_name: str,
    model_spec: Dict[str, str],
    output_dir: Path,
    samples: int,
    burn: int,
    thin: int,
    p_outlier: float
) -> Tuple[object, pd.DataFrame, Dict[str, object]]:
    """
    Instantiates and fits a single Bayesian HDDM regressor using MCMC sampling.

    Args:
        data (pd.DataFrame): The preprocessed dataset.
        model_name (str): Identifier for the specific model formulation.
        model_spec (Dict[str, str]): Formula definitions for 'a' and 'v'.
        output_dir (Path): Output directory for traces, stats, and models.
        samples (int): Total number of MCMC samples to draw.
        burn (int): Number of initial samples to discard as burn-in.
        thin (int): Thinning factor for sample retention.
        p_outlier (float): Assumed probability of outlier trials in the mixture model.

    Returns:
        Tuple[hddm.HDDMRegressor, pd.DataFrame, Dict]: The fitted model object, 
        posterior summary statistics, and model fit metrics (e.g., DIC).
    """
    output_dir = ensure_output_dir(output_dir)

    # Automatically enable group-only regressors for between-subject variables 
    # to prevent MCMC sampling convergence issues (deadlock).
    has_between_subj = "judgment_condition" in model_spec["a"] or "judgment_condition" in model_spec["v"]
    grp_only = model_spec.get("group_only_regressors", has_between_subj)
    
    print(f"Configuration Update: group_only_regressors enabled = {grp_only}")

    reg_descr = [
        {"model": model_spec["a"], "link_func": lambda x: x},
        {"model": model_spec["v"], "link_func": lambda x: x},
    ]

    model = hddm.HDDMRegressor(
        data,
        reg_descr,
        include=['v', 'a', 't'],
        p_outlier=p_outlier,
        group_only_regressors=grp_only,
    )

    print(f"\n=== Fitting Model: {model_name} ===")
    print(f"Boundary (a) Formula: {model_spec['a']}")
    print(f"Drift Rate (v) Formula: {model_spec['v']}")

    model.find_starting_values()
    dbname = str(output_dir / f"{model_name}_traces.db")
    
    # Execute MCMC Sampling
    model.sample(
        samples, 
        burn=burn, 
        thin=thin, 
        chains=4, 
        return_infdata=True,
        save_name=f"{output_dir}/{model_name}/{model_name}_fitted", 
        dbname=dbname, 
        db="pickle"
    )
    
    # Save Model Checkpoint
    model.save(str(output_dir / f"{model_name}.model"))
    
    # Extract and Export Posterior Statistics
    stats = model.gen_stats()
    stats.to_csv(output_dir / f"{model_name}_posterior_stats.csv")

    try:
        nodes_df = model.nodes_db.copy()
        if isinstance(nodes_df, pd.DataFrame):
            nodes_df.to_csv(output_dir / f"{model_name}_nodes_db.csv")
    except Exception as exc:
        print(f"Warning: Failed to export nodes_db for {model_name}. Exception: {exc}")

    # Isolate interaction coefficients for targeted analysis
    interaction_rows = stats[stats.index.str.contains(":", regex=False)].copy()
    if not interaction_rows.empty:
        interaction_rows.to_csv(output_dir / f"{model_name}_interaction_rows.csv")

    result = {
        "model_name": model_name,
        "a_formula": model_spec["a"],
        "v_formula": model_spec["v"],
        "hypothesis": model_spec["hypothesis"],
        "dic": float(model.dic),  # Deviance Information Criterion for model comparison
    }
    return model, stats, result


def fit_models_from_list(
    data: pd.DataFrame,
    model_space: Dict[str, Dict[str, str]],
    model_names: List[str],
    output_dir: Path,
    samples: int,
    burn: int,
    thin: int,
    p_outlier: float,
) -> Tuple[Dict[str, object], Dict[str, pd.DataFrame], pd.DataFrame]:
    """
    Iterates through a designated subset of the model space, fitting each model sequentially 
    and collating comparative fit indices.

    Args:
        data (pd.DataFrame): The preprocessed dataset.
        model_space (Dict): The full dictionary defining possible models.
        model_names (List[str]): Identifiers for the models to be fitted in this run.
        output_dir (Path): Directory for export outputs.
        samples (int): Total MCMC samples per model.
        burn (int): Burn-in samples per model.
        thin (int): Thinning factor per model.
        p_outlier (float): Assumed outlier probability for the mixture model.

    Returns:
        Tuple[Dict, Dict, pd.DataFrame]: Dictionaries storing model objects and posterior 
        stats, accompanied by a summarized DataFrame comparing models via Deviance 
        Information Criterion (DIC).
    """
    fitted_models = {}
    stats_dict = {}
    results = []

    for model_name in model_names:
        model, stats, result = fit_single_model(
            data=data,
            model_name=model_name,
            model_spec=model_space[model_name],
            output_dir=output_dir,
            samples=samples,
            burn=burn,
            thin=thin,
            p_outlier=p_outlier,
        )
        fitted_models[model_name] = model
        stats_dict[model_name] = stats
        results.append(result)

    # Compute comparative relative fit (Delta DIC)
    results_df = pd.DataFrame(results).sort_values("dic").reset_index(drop=True)
    results_df["delta_dic"] = results_df["dic"] - results_df["dic"].min()
    results_df.to_csv(Path(output_dir) / "model_comparison_dic.csv", index=False)
    
    return fitted_models, stats_dict, results_df

## 4. Create output folder


In [ ]:

OUTPUT_DIR = ensure_output_dir(OUTPUT_DIR)
print(f"Output directory: {OUTPUT_DIR}")


## 5. Load raw data

This cell reads the raw file and shows the first few rows.


In [ ]:

raw = read_data(INPUT_PATH)
print("Raw shape:", raw.shape)
print("Columns:", raw.columns.tolist())
raw.head()


## 6. Quick raw-data inspection

This cell helps you confirm:
- subject column
- raw levels of `judgment_condition`
- raw levels of `pair_type`
- raw values of `correctness`
- RT distribution before cleaning


In [ ]:

subject_col = pick_subject_column(raw)
print("Subject column:", subject_col)

print("\nRaw judgment_condition values:")
print(raw["judgment_condition"].value_counts(dropna=False))

print("\nRaw pair_type values:")
print(raw["pair_type"].value_counts(dropna=False))

print("\nRaw correctness values:")
print(raw["xab_resp"].value_counts(dropna=False))

print("\nRaw RT summary:")
print(pd.to_numeric(raw["xab_rt"], errors="coerce").describe())


## 7. Prepare HDDM input data

This step:
- recodes blank / missing `pair_type` to `None`
- drops invalid rows
- trims RT using `MIN_RT` and `MAX_RT`
- builds HDDM-friendly columns:
  - `subj_idx`
  - `rt`
  - `response`
  - `condition`


In [ ]:

hddm_df, summary = prepare_hddm_data(raw, MIN_RT, MAX_RT, OUTPUT_DIR)
print(json.dumps(summary, indent=2, ensure_ascii=False))
hddm_df.head()


## 8. Inspect cleaned data


In [ ]:

print("Cleaned shape:", hddm_df.shape)
print(hddm_df[["subj_idx", "judgment_condition", "pair_type", "response", "rt", "condition"]].head())

print("\nCondition counts:")
print(hddm_df["condition"].value_counts().sort_index())

print("\nSubjects per condition:")
print(
    hddm_df.groupby(["judgment_condition", "pair_type"], observed=True)["subj_idx"]
    .nunique()
    .reset_index(name="n_subjects")
)


## 9. Compute and save descriptives

This creates summary tables for:
- counts
- accuracy
- correct-trial RT


In [ ]:

DESC = compute_descriptives(hddm_df)
save_descriptives(DESC, OUTPUT_DIR)

print("Counts:")
print(DESC["counts"])

print("\nAccuracy:")
print(DESC["subj_acc"])

print("\nCorrect RT:")
print(DESC["subj_rt_correct"])


## 16. Choose factor reference levels

You can either let the notebook infer them from the cleaned data, or force them manually in the settings cell.


In [ ]:
judgment_levels = ordered_present_levels(hddm_df["judgment_condition"], PREFERRED_JUDGMENT_ORDER)
pair_levels = ordered_present_levels(hddm_df["pair_type"], PREFERRED_PAIR_ORDER)


judgment_ref = MANUAL_JUDGMENT_REFERENCE if MANUAL_JUDGMENT_REFERENCE is not None else judgment_levels[0]
pair_ref = MANUAL_PAIR_REFERENCE if MANUAL_PAIR_REFERENCE is not None else choose_pair_reference(pair_levels)

print("judgment_levels:", judgment_levels)
print("pair_levels:", pair_levels)
print("judgment_ref:", judgment_ref)
print("pair_ref:", pair_ref)

## 17. Build model space

### Main theoretical logic
- `M4`: interaction in **v**
- `M5`: interaction in **a**
- `M8`: interaction in **both a and v**

The other models serve as simpler comparison baselines.


In [ ]:

# Use the updated variable names: judgment_ref and pair_ref
MODEL_SPACE = build_model_space(judgment_ref=judgment_ref, pair_ref=pair_ref)

model_table = pd.DataFrame(
    [
        {
            "model_name": name,
            "a_formula": spec["a"],
            "v_formula": spec["v"],
            "hypothesis": spec["hypothesis"],
        }
        for name, spec in MODEL_SPACE.items()
    ]
)
model_table

## 19. Fit one test model first

Run this before anything else to confirm that:
- HDDM imports correctly
- your data format is acceptable
- sampling works in your environment


In [ ]:
%%capture
TEST_MODEL_NAME = "M4_v_judgment_x_pair"
test_model, test_stats, test_result = fit_single_model(
    data=hddm_df,
    model_name=TEST_MODEL_NAME,
    model_spec=MODEL_SPACE[TEST_MODEL_NAME],
    output_dir=OUTPUT_DIR / "test_model",
    samples=SAMPLES_TEST,
    burn=BURN_TEST,
    thin=THIN_TEST,
    p_outlier=P_OUTLIER,
)

print(test_result)
test_stats.head()
test_model.dic

## 20. Inspect posterior stats for the test model

Look especially at:
- interaction terms containing `:`
- whether coefficients for `v` or `a` move away from zero
- uncertainty intervals


In [ ]:

interaction_rows_test = test_stats[test_stats.index.str.contains(":", regex=False)].copy()
interaction_rows_test if not interaction_rows_test.empty else test_stats.head(20)


## 22. Fit a small subset of models

This is the most useful intermediate step before running the full model space.


In [ ]:
%%capture
SUBSET_MODELS = [
    "M0_baseline",
     "M1_v_train_main",
    "M8_av_judgment_x_pair"
]

subset_models, subset_stats, subset_results = fit_models_from_list(
    data=hddm_df,
    model_space=MODEL_SPACE,
    model_names=SUBSET_MODELS,
    output_dir=OUTPUT_DIR / "formal_models",
    samples=SAMPLES_FULL,
    burn=BURN_FULL,
    thin=THIN_FULL,
    p_outlier=P_OUTLIER,
)

subset_results

## 23. Compare the subset by DIC


In [ ]:

subset_results.sort_values("dic").reset_index(drop=True)


In [ ]:
%%capture
SUBSET_MODELS = [
    "M0_baseline",
    "M4_v_judgment_x_pair",
    "M5_a_judgment_x_pair",
    "M8_av_judgment_x_pair",
]

subset_models, subset_stats, subset_results = fit_models_from_list(
    data=hddm_df,
    model_space=MODEL_SPACE,
    model_names=SUBSET_MODELS,
    output_dir=OUTPUT_DIR / "subfullset_models_thin=2",
    samples=SAMPLES_FULL,
    burn=BURN_FULL,
    thin=2,
    p_outlier=P_OUTLIER,
)

subset_results

## 24. Fit the full model space

Only run this after the test model and subset both work.


In [ ]:
%%capture
FULL_MODEL_NAMES = list(MODEL_SPACE.keys())

full_models, full_stats, full_results = fit_models_from_list(
    data=hddm_df,
    model_space=MODEL_SPACE,
    model_names=FULL_MODEL_NAMES,
    output_dir=OUTPUT_DIR / "full_models",
    samples=SAMPLES_FULL,
    burn=BURN_FULL,
    thin=THIN_FULL,
    p_outlier=P_OUTLIER,
)

full_results


## 25. Compare all models by DIC

Smaller DIC is better.
The most important comparison is whether the winning model places the interaction in:
- `v`
- `a`
- or both


In [ ]:

full_results = full_results.sort_values("dic").reset_index(drop=True)
full_results


## 26. Identify the winning model


In [ ]:

best_model_name = full_results.iloc[0]["model_name"]
print("Best model:", best_model_name)
print(MODEL_SPACE[best_model_name]["hypothesis"])


## 27. Inspect posterior stats for the winning model


In [ ]:

best_stats = full_stats[best_model_name]
best_stats.head(30)


## 28. Extract interaction rows from the winning model

These rows are usually the most directly relevant to your main theoretical question.


In [ ]:

best_interactions = best_stats[best_stats.index.str.contains(":", regex=False)].copy()
best_interactions if not best_interactions.empty else best_stats.head(30)


## 29. Optional: save a concise winning-model summary


In [ ]:

winning_summary = {
    "best_model_name": best_model_name,
    "hypothesis": MODEL_SPACE[best_model_name]["hypothesis"],
    "a_formula": MODEL_SPACE[best_model_name]["a"],
    "v_formula": MODEL_SPACE[best_model_name]["v"],
    "best_dic": float(full_results.iloc[0]["dic"]),
}
with open(OUTPUT_DIR / "winning_model_summary.json", "w", encoding="utf-8") as f:
    json.dump(winning_summary, f, indent=2, ensure_ascii=False)

winning_summary


## 30. Interpretation notes

Use this section as a reading guide when you inspect the winning model.

### If `M4_v_repeat_x_noise` wins
The interaction is mainly expressed in **drift rate (`v`)**.

### If `M5_a_repeat_x_noise` wins
The interaction is mainly expressed in **boundary separation (`a`)**.

### If `M8_av_repeat_x_noise` wins
The interaction is jointly expressed in **both `a` and `v`**.
